# Test TSE model

# Сравнение TSE-моделей на записи лекции

Блокнот прогоняет один mix через TSE-модели и сохраняет результаты в wav для прослушивания.

**Модели:**
1. USEF-TFGridNet (8 кГц, с ресемплингом)
2. USEF-SepFormer (8 кГц, с ресемплингом)
3. TSE Pos-Neg Enroll — proposed-monaural (16 кГц)
4. TSE Pos-Neg Enroll — improved-monaural (16 кГц)
5. MeanFlow-TSE (16 кГц)
6. TSELM (16 кГц) — заготовка

## 0. Пути и утилиты

In [2]:
from pathlib import Path
import sys
import torch
import torchaudio
import torchaudio.functional as AF

# === Корни ===
PROJECT_ROOT = Path.cwd()
MODELS_DIR   = PROJECT_ROOT / 'models'

# === Модели ===
USEF_REPO           = MODELS_DIR / 'USEF-TSE-main'
USEF_TFGRIDNET_DIR  = USEF_REPO  / 'chkpt' / 'USEF-TFGridNet'
USEF_SEPFORMER_DIR  = USEF_REPO  / 'chkpt' / 'USEF-SepFormer'

POSNEG_REPO      = MODELS_DIR / 'PosNegEnrollment'
POSNEG_CKPT_DIR  = POSNEG_REPO / 'chkpt'

MEANFLOW_REPO     = MODELS_DIR / 'MeanFlow-TSE-main'
MEANFLOW_CKPT_DIR = MEANFLOW_REPO / 'chkpt'

TSELM_BASE = MODELS_DIR / 'TSELM'
TSELM_REPO = TSELM_BASE / 'TSELM-main'

# === Вход/выход ===
IO_DIR   = PROJECT_ROOT / 'output'
MIX_PATH = IO_DIR / 'input_30min.wav'                  # Входной файл
REF_PATH = IO_DIR / 'refs' / 'ref_000.wav' # reference препода
REF_CLEAN_PATH = IO_DIR / 'refs' / 'ref_000_clean.wav' # очищенный референс, 8k
POS_PATH = IO_DIR / 'refs' / 'pos_enroll.wav'    # заполняется в секции 0.1
NEG_PATH = IO_DIR / 'refs' / 'neg_enroll.wav'    # заполняется в секции 0.1

OUT_DIR = IO_DIR / 'tse_out'
OUT_DIR.mkdir(exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


In [3]:
def load_wav(path, target_sr=16000):
    """(1, T) float32 mono tensor с ресемплом."""
    wav, sr = torchaudio.load(str(path))
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != target_sr:
        wav = AF.resample(wav, sr, target_sr)
    return wav

def save_wav(wav, path, sr=16000):
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    wav = wav.detach().cpu().to(torch.float32)
    peak = wav.abs().max()
    if peak > 1.0:
        wav = wav / peak * 0.99
    torchaudio.save(str(path), wav, sr)
    print(f'Saved: {path.relative_to(PROJECT_ROOT)}')

# Быстрая проверка входов
for p in [MIX_PATH, REF_PATH, POS_PATH, NEG_PATH]:
    print(f'{p.relative_to(PROJECT_ROOT)}: {"OK" if p.exists() else "MISSING"}')

output\input_30min.wav: OK
output\refs\ref_000.wav: OK
output\refs\pos_enroll.wav: OK
output\refs\neg_enroll.wav: OK


In [4]:
import gc, sys, torch

def hard_reset_memory():
    """Тотальная очистка GPU + Python-ссылок на модели."""
    # Убиваем переменные моделей из globals
    for name in ['model', 'meanflow', 't_predicter', 'encoder', 'encoder_head',
                 'est', 'out', 'mix', 'ref', 'pos', 'neg', 'cond_emb', 
                 'init_state', 'mix_spec', 'ref_spec', 'src_hat', 'src_hat_spec',
                 'mix_spec_pad', 'chunk']:
        if name in globals():
            del globals()[name]
    
    # Чистим traceback IPython (держит тензоры последней ошибки)
    sys.last_traceback = None
    sys.last_type = None
    sys.last_value = None
    
    # GC + CUDA cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print(f'Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
        print(f'Reserved:  {torch.cuda.memory_reserved()  / 1e9:.2f} GB')

hard_reset_memory()

Allocated: 0.00 GB
Reserved:  0.00 GB


In [5]:
# === Глобальная очистка окружения — запускай перед переключением между моделями ===
def reset_env():
    import gc, sys, torch
    
    # Чистим переменные моделей
    for name in ['model', 'meanflow', 't_predicter', 'encoder', 'encoder_head',
                 'est', 'out', 'mix', 'ref', 'pos', 'neg', 'cond_emb', 
                 'init_state', 'mix_spec', 'ref_spec', 'src_hat', 'src_hat_spec',
                 'mix_wav', 'ref_wav', 'ref_fit', 'result', 'outputs']:
        if name in globals():
            del globals()[name]
    
    # Чистим sys.modules от коллидирующих имён между репами моделей
    for mod in list(sys.modules.keys()):
        prefixes = ('utils', 'models', 'dataset', 'exp', 'scheduler',
                    'model.', 'improved_model.', 'train_meanflow', 'meanflow')
        if any(mod == p.rstrip('.') or mod.startswith(p) for p in prefixes):
            del sys.modules[mod]
    
    # Чистим IPython traceback
    sys.last_traceback = None
    sys.last_type = None
    sys.last_value = None
    
    # GC + CUDA
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    
    print(f'Reset done. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
          f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved')

# Вызов:
# reset_env()

## 0.1 Подготовка Pos/Neg enrollments (один раз)

Для Pos-Neg моделей нужны **два** коротких куска (~3 сек каждый):
- **pos_enroll** — препод говорит
- **neg_enroll** — препод молчит, говорит студент / аудитория


In [ ]:
# Впиши тайм-коды в секундах. Длина ~3 сек оптимальна (как при обучении модели).
POS_START, POS_END = 0, 3   # препод говорит
NEG_START, NEG_END = 1205.28, 1208.28   # препод молчит

def cut_and_save(src_wav_path, t_start, t_end, dst_path, sr=16000):
    wav = load_wav(src_wav_path, target_sr=sr)  # (1, T)
    s, e = int(t_start * sr), int(t_end * sr)
    chunk = wav[:, s:e]
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    save_wav(chunk.squeeze(0), dst_path, sr=sr)
    print(f'  Length: {(e-s)/sr:.2f} s')

cut_and_save(REF_PATH, POS_START, POS_END, POS_PATH)
cut_and_save(MIX_PATH, NEG_START, NEG_END, NEG_PATH)

---
## 1. USEF-TFGridNet (8 кГц)

Конфиг: `models/USEF-TSE-main/chkpt/USEF-TFGridNet/config.yaml`  
Веса: `models/USEF-TSE-main/chkpt/USEF-TFGridNet/wsj0-2mix/temp_best.pth.tar`  


In [ ]:
from collections import OrderedDict
from hyperpyyaml import load_hyperpyyaml

if str(USEF_REPO) not in sys.path:
    sys.path.insert(0, str(USEF_REPO))

def load_usef_model(config_path, ckpt_path, device):
    with open(config_path, 'r') as f:
        config = load_hyperpyyaml(f.read())
    model = config['modules']['masknet']
    info = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    sd = OrderedDict()
    for k, v in info['model_state_dict'].items():
        name = k.replace('module.', '').replace('convolution_', 'convolution_module.')
        sd[name] = v
    model.load_state_dict(sd)
    return model.to(device).eval(), config['sample_rate']


def run_usef_chunked(model, mix, ref, chunk_sec, sr):
    """Прогон по не-перекрывающимся чанкам. Возвращает (T,) на CPU."""
    chunk_samples = int(chunk_sec * sr)
    total = mix.shape[-1]
    outputs = []
    
    with torch.no_grad():
        for start in range(0, total, chunk_samples):
            end = min(start + chunk_samples, total)
            chunk = mix[:, start:end]
            est = model(chunk, ref).squeeze().cpu()
            outputs.append(est)
            pct = end / total * 100
            print(f'  {start/sr:6.1f}-{end/sr:6.1f}s  ({pct:5.1f}%)', flush=True)
            
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
    
    return torch.cat(outputs, dim=-1)

In [ ]:
model, model_sr = load_usef_model(
    config_path = USEF_TFGRIDNET_DIR / 'config.yaml',
    ckpt_path   = USEF_TFGRIDNET_DIR / 'wsj0-2mix' / 'temp_best.pth.tar',
    device      = DEVICE,
)
print(f'Native SR: {model_sr}')

mix = load_wav(MIX_PATH, target_sr=model_sr).to(DEVICE)
ref = load_wav(REF_CLEAN_PATH, target_sr=model_sr).to(DEVICE)

est = run_usef_chunked(model, mix, ref, chunk_sec=15, sr=model_sr)

save_wav(est, OUT_DIR / 'usef_tfgridnet_8k.wav', sr=model_sr)
est_16k = AF.resample(est.unsqueeze(0), model_sr, 16000).squeeze()
save_wav(est_16k, OUT_DIR / 'usef_tfgridnet_16k.wav', sr=16000)

del model
if DEVICE == 'cuda': torch.cuda.empty_cache()

---
## 2. USEF-SepFormer (8 кГц)

In [ ]:
model, model_sr = load_usef_model(
    config_path = USEF_SEPFORMER_DIR / 'config.yaml',
    ckpt_path   = USEF_SEPFORMER_DIR / 'wsj0-2mix' / 'temp_best.pth.tar',
    device      = DEVICE,
)

mix = load_wav(MIX_PATH, target_sr=model_sr).to(DEVICE)
ref = load_wav(REF_PATH, target_sr=model_sr).to(DEVICE)

est = run_usef_chunked(model, mix, ref, chunk_sec=8, sr=model_sr)

save_wav(est, OUT_DIR / 'usef_sepformer_8k.wav', sr=model_sr)
est_16k = AF.resample(est.cpu().unsqueeze(0), model_sr, 16000).squeeze()
save_wav(est_16k, OUT_DIR / 'usef_sepformer_16k.wav', sr=16000)

del model
if DEVICE == 'cuda': torch.cuda.empty_cache()

---
## 3. TSE Pos-Neg Enroll — proposed-monaural (16 кГц)

Архитектура из статьи: `TFGridNet_origcrossattn_causal_single_emb` (causal).  
Параметры модели взяты дословно из `eval_monaural.py`.

In [ ]:
if str(POSNEG_REPO) not in sys.path:
    sys.path.insert(0, str(POSNEG_REPO))

from models.PosNegEnrollment.model.tfgridnet_encoder import TFGridNet_encoder
from models.PosNegEnrollment.model.GridnetAttnHead import GridNetBlock_attnhead
from models.PosNegEnrollment.model.tfgridnet_KVfusion import TFGridNet_KVfusion
from models.PosNegEnrollment.model.tfgridnet_crossattn_causal_single_emb import TFGridNet_origcrossattn_causal_single_emb

def run_posneg_chunked(model, mix, cond_emb, chunk_sec=10, sr=16000):
    """Causal chunking для Pos-Neg proposed. mix: (1, 1, T)."""
    chunk_samples = int(chunk_sec * sr)
    total = mix.shape[-1]
    outputs = []
    state = model.init_buffers(mix.shape[0], mix.device)
    
    with torch.no_grad():
        for start in range(0, total, chunk_samples):
            end = min(start + chunk_samples, total)
            chunk = mix[:, :, start:end]
            out, state = model(chunk, cond_emb, state)  # state передаётся
            outputs.append(out.squeeze().cpu())
            print(f'  {start/sr:6.1f}-{end/sr:6.1f}s ({end/total*100:.1f}%)', flush=True)
            
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    return torch.cat(outputs, dim=-1)


encoder = TFGridNet_encoder(num_ch=2, n_fft=128, stride=64, num_blocks=3, binaural=False)
encoder_head = GridNetBlock_attnhead(layer_num=2, pooling_size=1, stride=1)

model = TFGridNet_origcrossattn_causal_single_emb(
    n_fft=128, stride=64, n_layers=3, lstm_hidden_units=64,
    emb_dim=64, emb_ks=1,
    model_normalize=True,
    Fusion_class=TFGridNet_KVfusion,
    pooling_size=40, fusion_stride=40,
    encoder=encoder, encoder_head=encoder_head,
    train_encoder=False, train_encoder_head=False,
    fusion_layer=[0, 1],
    binaural=False,
).to(DEVICE)

state = torch.load(POSNEG_CKPT_DIR / 'proposed-monaural.pt', map_location='cpu', weights_only=False)
model.load_state_dict(state['state_dict'], strict=True)
model.eval()

# Форма (1, 1, T)
mix = load_wav(MIX_PATH, target_sr=16000).unsqueeze(0).to(DEVICE)
pos = load_wav(POS_PATH, target_sr=16000).unsqueeze(0).to(DEVICE)
neg = load_wav(NEG_PATH, target_sr=16000).unsqueeze(0).to(DEVICE)

# Pad до 48000 сэмплов (3 сек) если короче
def pad_to(x, n):
    d = n - x.shape[-1]
    return torch.nn.functional.pad(x, (0, d)) if d > 0 else x
pos = pad_to(pos, 48000)
neg = pad_to(neg, 48000)

with torch.no_grad():
    cond_emb = model.encode(pos, neg)

out = run_posneg_chunked(model, mix, cond_emb, chunk_sec=10, sr=16000)
save_wav(out, OUT_DIR / 'posneg_proposed.wav', sr=16000)


del model
if DEVICE == 'cuda': torch.cuda.empty_cache()

---
## 4. TSE Pos-Neg Enroll — improved-monaural (16 кГц)

Вариант с USEF-TFGridNet fusion (лучше по метрикам). Модуль `improved_model/USEF_TFGridnet.py`.

In [ ]:
# В improved варианте encoder — из 'model', а head и USEF — из 'improved_model'
from models.PosNegEnrollment.model.tfgridnet_encoder import TFGridNet_encoder as TFGridNet_encoder_v2
from models.PosNegEnrollment.improved_model.GridnetAttnHead import GridNetBlock_attnhead as GridNetBlock_attnhead_imp
from models.PosNegEnrollment.improved_model.USEF_TFGridnet import Tar_Model


def run_posneg_improved_chunked(model, mix, cond_emb, chunk_sec=10, sr=16000):
    chunk_samples = int(chunk_sec * sr)
    total = mix.shape[-1]
    outputs = []
    with torch.no_grad():
        for start in range(0, total, chunk_samples):
            end = min(start + chunk_samples, total)
            chunk = mix[:, :, start:end]
            out = model(chunk, cond_emb).squeeze().cpu()
            outputs.append(out)
            print(f'  {start/sr:6.1f}-{end/sr:6.1f}s ({end/total*100:.1f}%)', flush=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return torch.cat(outputs, dim=-1)


encoder = TFGridNet_encoder_v2(num_ch=2, n_fft=128, stride=64, num_blocks=1, binaural=False)
encoder_head = GridNetBlock_attnhead_imp(
    layer_num=2, pooling_size=1, stride=1,
    return_clean_dvec=False, out_dim=0,
    refine_layer_num=2, fusion_shortcut=[0, 1], cut_pos=True,
)

model = Tar_Model(
    n_freqs=65, hidden_channels=64, n_head=4,
    emb_dim=64, emb_ks=1, emb_hs=1, num_layers=3,
    encoder=encoder, encoder_head=encoder_head,
    train_encoder=False, train_encoder_head=False,
).to(DEVICE)

state = torch.load(POSNEG_CKPT_DIR / 'improved-monaural.pt', map_location='cpu', weights_only=False)
model.load_state_dict(state['state_dict'], strict=True)
model.eval()

mix = load_wav(MIX_PATH, target_sr=16000).unsqueeze(0).to(DEVICE)
pos = load_wav(POS_PATH, target_sr=16000).unsqueeze(0).to(DEVICE)
neg = load_wav(NEG_PATH, target_sr=16000).unsqueeze(0).to(DEVICE)
pos = pad_to(pos, 48000)
neg = pad_to(neg, 48000)

with torch.no_grad():
    cond_emb, pos_std, dec_audio = model.encoder_pos_neg(pos, neg, recons=False)
    
out = run_posneg_improved_chunked(model, mix, cond_emb, chunk_sec=10, sr=16000)
save_wav(out, OUT_DIR / 'posneg_improved.wav', sr=16000)

del model
if DEVICE == 'cuda': torch.cuda.empty_cache()

---
## 5. MeanFlow-TSE (16 кГц)

Flow-matching, one-step inference. Внутренне чанкуется на 3-секундные STFT-куски.

In [6]:
if str(MEANFLOW_REPO) not in sys.path:
    sys.path.insert(0, str(MEANFLOW_REPO))

import pytorch_lightning as pl

from train_meanflow import LightningModule as MeanFlowLightningModule, parse_config as parse_meanflow_config
from models.t_predicter import TPredicter
from utils.transforms import stft_torch, istft_torch

PREDICTOR_TYPE = 'ECAPAMLP'   # или 'HALF' — тогда не нужен t-predictor
NUM_STEPS      = 1

config = parse_meanflow_config(str(MEANFLOW_REPO / 'config' / 'config_MeanFlowTSE_clean.yaml'))
pl.seed_everything(config['seed'])

meanflow = MeanFlowLightningModule.load_from_checkpoint(
    str(MEANFLOW_CKPT_DIR / 'best-clean-weights.ckpt'), config=config
).to(DEVICE).eval()

t_predicter = None
if PREDICTOR_TYPE == 'ECAPAMLP':
    t_predicter = TPredicter(**config['t_predicter'])
    ck = torch.load(MEANFLOW_CKPT_DIR / 't-predictor-clean-weights.ckpt', map_location=DEVICE)
    sd = {k[6:] if k.startswith('model.') else k: v for k, v in ck['state_dict'].items()}
    t_predicter.load_state_dict(sd)
    t_predicter = t_predicter.to(DEVICE).eval()

n_fft      = config['dataset']['n_fft']
hop_length = config['dataset']['hop_length']
win_length = config['dataset']['win_length']
SR         = config['dataset']['sample_rate']
print(f'SR={SR}, n_fft={n_fft}, hop={hop_length}, win={win_length}')

c:\Users\matve\Documents\diplom\python_SSLR\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 42


attention mode is flash
SR=16000, n_fft=510, hop=128, win=510


In [7]:
# Helpers из inference_sample.ipynb
def pad_and_reshape(tensor, multiple):
    n, d, l = tensor.shape
    pad_len = (multiple - (l % multiple)) % multiple
    padded = torch.nn.functional.pad(tensor, (0, pad_len))
    return torch.cat(torch.chunk(padded, padded.shape[-1] // multiple, dim=-1), dim=0), l

def reshape_and_remove_padding(tensor, original_length):
    n_k, d, m = tensor.shape
    n = original_length // m + (1 if original_length % m != 0 else 0)
    return torch.cat(torch.chunk(tensor, n, dim=0), dim=-1)[:, :, :original_length]

def sample_euler_multistep(model, mixture_spec, enrollment_spec, alpha, num_steps=1):
    bs = mixture_spec.size(0)
    dev = mixture_spec.device
    if not torch.is_tensor(alpha): alpha = torch.tensor([alpha], device=dev)
    if alpha.ndim == 0: alpha = alpha.unsqueeze(0)
    if alpha.shape[0] == 1 and bs > 1: alpha = alpha.repeat(bs)
    z = mixture_spec
    t_grid = torch.linspace(alpha.mean().item(), 1.0, num_steps + 1, device=dev)
    for step in range(num_steps):
        t_cur = alpha.clone() if step == 0 else torch.full((bs,), t_grid[step].item(), device=dev)
        t_tgt = torch.full((bs,), t_grid[step + 1].item(), device=dev)
        dt = t_tgt - t_cur
        v = model(z, t_cur, t_tgt, enrollment_spec)
        z = z + dt.view(bs, 1, 1) * v
    return z

def scale_audio(audio):
    m = audio.abs().max()
    return audio / m if m > 1.0 else audio

In [8]:
BATCH_SIZE = 4  # сколько 3-секундных чанков прогоняем за раз. Начни с 4, можешь увеличить если ОК

mix_wav = load_wav(MIX_PATH, target_sr=SR).to(DEVICE)
ref_wav = load_wav(REF_PATH, target_sr=SR).to(DEVICE)
ref_wav = ref_wav[:, :SR * 10]
mix_mono = mix_wav.squeeze(0)
ref_mono = ref_wav.squeeze(0)

mix_spec = stft_torch(mix_mono, n_fft=n_fft, hop_length=hop_length, win_length=win_length).unsqueeze(0)
ref_spec = stft_torch(ref_mono, n_fft=n_fft, hop_length=hop_length, win_length=win_length).unsqueeze(0)
print('With start')
with torch.no_grad():
    # --- alpha ---
    if PREDICTOR_TYPE == 'ECAPAMLP':
        mix_excerpt = mix_wav[:, :SR * 60]   # 10 секунд mix
        ref_excerpt = ref_wav[:, :SR * 10]   # 10 секунд ref (если короче — берёт сколько есть)
        alpha = t_predicter(mix_excerpt, ref_excerpt, aug=False)
    else:
        alpha = torch.tensor([0.5], device=DEVICE)
    print(f'alpha = {alpha.item():.4f}')

    # --- чанкинг STFT ---
    multiple = SR * 3 // hop_length + 1
    mix_spec_pad, orig_len = pad_and_reshape(mix_spec, multiple)
    total_chunks = mix_spec_pad.shape[0]
    print(f'Total chunks: {total_chunks}, batch_size: {BATCH_SIZE}')

    # --- прогон по micro-batch'ам ---
    out_chunks = []
    for i in range(0, total_chunks, BATCH_SIZE):
        mini = mix_spec_pad[i:i + BATCH_SIZE].float()
        bs_mini = mini.shape[0]
        ref_rep = ref_spec.repeat(bs_mini, 1, 1)
        alpha_mini = alpha.repeat(bs_mini) if alpha.shape[0] == 1 else alpha[i:i + BATCH_SIZE]

        out_mini = sample_euler_multistep(
            model=meanflow.model,
            mixture_spec=mini,
            enrollment_spec=ref_rep,
            alpha=alpha_mini,
            num_steps=NUM_STEPS,
        )
        out_chunks.append(out_mini.cpu())  # сразу на CPU, чтоб не копить в VRAM
        print(f'  chunk {i+bs_mini}/{total_chunks}', flush=True)
        torch.cuda.empty_cache()

    src_hat_spec = torch.cat(out_chunks, dim=0).to(DEVICE)
    src_hat_spec = reshape_and_remove_padding(src_hat_spec, orig_len)

    src_hat = istft_torch(
        src_hat_spec,
        n_fft=n_fft, hop_length=hop_length, win_length=win_length,
        length=mix_mono.shape[-1],
    )
    src_hat = scale_audio(src_hat.cpu())

save_wav(src_hat.squeeze(), OUT_DIR / 'meanflow.wav', sr=SR)

del meanflow
if t_predicter is not None: del t_predicter
if DEVICE == 'cuda': torch.cuda.empty_cache()

With start
alpha = 0.7193
Total chunks: 599, batch_size: 4
  chunk 4/599
  chunk 8/599
  chunk 12/599
  chunk 16/599
  chunk 20/599
  chunk 24/599
  chunk 28/599
  chunk 32/599
  chunk 36/599
  chunk 40/599
  chunk 44/599
  chunk 48/599
  chunk 52/599
  chunk 56/599
  chunk 60/599
  chunk 64/599
  chunk 68/599
  chunk 72/599
  chunk 76/599
  chunk 80/599
  chunk 84/599
  chunk 88/599
  chunk 92/599
  chunk 96/599
  chunk 100/599
  chunk 104/599
  chunk 108/599
  chunk 112/599
  chunk 116/599
  chunk 120/599
  chunk 124/599
  chunk 128/599
  chunk 132/599
  chunk 136/599
  chunk 140/599
  chunk 144/599
  chunk 148/599
  chunk 152/599
  chunk 156/599
  chunk 160/599
  chunk 164/599
  chunk 168/599
  chunk 172/599
  chunk 176/599
  chunk 180/599
  chunk 184/599
  chunk 188/599
  chunk 192/599
  chunk 196/599
  chunk 200/599
  chunk 204/599
  chunk 208/599
  chunk 212/599
  chunk 216/599
  chunk 220/599
  chunk 224/599
  chunk 228/599
  chunk 232/599
  chunk 236/599
  chunk 240/599
  chunk

---
## 6. TSELM (16 кГц)

Все подмодели на месте:
- Основной: `models/TSELM/tselm_l.pth` (3.2 ГБ)
- WavLM-Large: `models/TSELM/wavlm-large/pytorch_model.bin`
- k-means: `models/TSELM/kmeans_ckpt/*.pt`
- HiFi-GAN: `models/TSELM/hifigan-wavlm-.../generator.ckpt`

Как дойдёшь — кинь `models/TSELM/TSELM-main/inference.py`, заполню.

In [4]:
# === Секция 6: TSELM ===
import gc

# --- Освободить VRAM перед загрузкой (TSELM тяжёлый) ---
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

TSELM_CKPT   = TSELM_BASE / 'tselm_l.pth'
TSELM_CONFIG = TSELM_REPO / 'config' / 'tselm_l.yaml'

WAVLM_DIR   = TSELM_BASE / 'wavlm-large'
KMEANS_DIR  = TSELM_BASE / 'kmeans_ckpt'
HIFIGAN_DIR = TSELM_BASE / 'hifigan-wavlm-l1-3-7-12-18-23-k1000-LibriTTS'

if str(TSELM_REPO) not in sys.path:
    sys.path.insert(0, str(TSELM_REPO))

from hyperpyyaml import load_hyperpyyaml

# --- Грузим конфиг с подставленными путями ---
with open(TSELM_CONFIG, 'r') as f:
    config_text = f.read()

overrides = {
    'wavlm_path':   str(WAVLM_DIR).replace('\\', '/'),
    'kmeans_path':  str(KMEANS_DIR).replace('\\', '/'),
    'hifi_gan_path': str(HIFIGAN_DIR).replace('\\', '/'),
    # Dummy-пути для полей, которые не используются при инференсе, но требуются конфигом
    'tr_data_scp_path': 'dummy',
    'cv_mix_path':      'dummy',
    'cv_ref_path':      'dummy',
    'cv_clean_path':    'dummy',
}

print('Loading TSELM config (это может занять 1-2 минуты — грузятся WavLM и HiFi-GAN)...', flush=True)
config = load_hyperpyyaml(config_text, overrides=overrides)
model = config['model']

print('Loading TSELM checkpoint (3.2 GB)...', flush=True)
ckpt = torch.load(TSELM_CKPT, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'], strict=False)
model = model.to(DEVICE).eval()
print(f'Model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB', flush=True)

Loading TSELM config (это может занять 1-2 минуты — грузятся WavLM и HiFi-GAN)...


c:\Users\matve\Documents\diplom\python_SSLR\.venv\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
c:\Users\matve\Documents\diplom\python_SSLR\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at c:/Users/matve/Documents/diplom/python_SSLR/models/TSELM/wavlm-large were not used when initializing WavLMModel: ['encoder.pos_conv_embed.conv.weight_g', 'encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing WavLMModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This

Loading TSELM checkpoint (3.2 GB)...
Model loaded. VRAM: 2.06 GB


In [6]:
# === TSELM inference ===
import time

SR_TSELM = 16000
# Длина mix-чанка: модель обучалась на 48080 сэмплов = 3.005 сек.
# Reference длины: 64080 сэмплов = 4.005 сек.
MIX_CHUNK_SAMPLES = 48080  
REF_LEN_SAMPLES   = 64080

# --- Загрузка аудио ---
mix_wav = load_wav(MIX_PATH, target_sr=SR_TSELM).to(DEVICE)  # (1, T)
ref_wav = load_wav(REF_PATH, target_sr=SR_TSELM).to(DEVICE)

# Reference обрезаем/паддим до 4 сек (как при обучении)
def fit_length(x, target_len):
    T = x.shape[-1]
    if T >= target_len:
        return x[..., :target_len]
    return torch.nn.functional.pad(x, (0, target_len - T))

ref_fit = fit_length(ref_wav, REF_LEN_SAMPLES)   # (1, 64080)
print(f'mix: {mix_wav.shape}, ref: {ref_fit.shape}')

# --- Чанкинг mix с прокидыванием через model.inference ---
total = mix_wav.shape[-1]
outputs = []

t_start = time.time()
with torch.no_grad():
    for start in range(0, total, MIX_CHUNK_SAMPLES):
        end = min(start + MIX_CHUNK_SAMPLES, total)
        chunk = mix_wav[:, start:end]  # (1, T_chunk)
        
        # padим последний чанк до полной длины, чтобы модели было комфортно
        orig_chunk_len = chunk.shape[-1]
        if orig_chunk_len < MIX_CHUNK_SAMPLES:
            chunk = fit_length(chunk, MIX_CHUNK_SAMPLES)
        
        # model.inference(mix, regi) → (output, _)
        out, _ = model.inference(chunk, ref_fit)
        out = out[:, :orig_chunk_len]  # обрезаем padding назад
        outputs.append(out.cpu().squeeze())
        
        elapsed = time.time() - t_start
        vram = torch.cuda.memory_allocated() / 1e9
        print(f'  {start/SR_TSELM:6.1f}-{end/SR_TSELM:6.1f}s  '
              f'elapsed: {elapsed:5.1f}s  VRAM: {vram:.2f} GB', flush=True)
        
        torch.cuda.empty_cache()

result = torch.cat(outputs, dim=-1)
save_wav(result, OUT_DIR / 'tselm.wav', sr=SR_TSELM)

del model
gc.collect()
torch.cuda.empty_cache()
print(f'Total: {time.time()-t_start:.1f}s')

mix: torch.Size([1, 28800000]), ref: torch.Size([1, 64080])
     0.0-   3.0s  elapsed:   1.6s  VRAM: 2.25 GB
     3.0-   6.0s  elapsed:   2.0s  VRAM: 2.25 GB
     6.0-   9.0s  elapsed:   2.4s  VRAM: 2.25 GB
     9.0-  12.0s  elapsed:   2.8s  VRAM: 2.25 GB
    12.0-  15.0s  elapsed:   3.2s  VRAM: 2.25 GB
    15.0-  18.0s  elapsed:   3.6s  VRAM: 2.25 GB
    18.0-  21.0s  elapsed:   4.0s  VRAM: 2.25 GB
    21.0-  24.0s  elapsed:   4.4s  VRAM: 2.25 GB
    24.0-  27.0s  elapsed:   4.9s  VRAM: 2.25 GB
    27.0-  30.1s  elapsed:   5.3s  VRAM: 2.25 GB
    30.1-  33.1s  elapsed:   5.7s  VRAM: 2.25 GB
    33.1-  36.1s  elapsed:   6.1s  VRAM: 2.25 GB
    36.1-  39.1s  elapsed:   6.5s  VRAM: 2.25 GB
    39.1-  42.1s  elapsed:   6.9s  VRAM: 2.25 GB
    42.1-  45.1s  elapsed:   7.3s  VRAM: 2.25 GB
    45.1-  48.1s  elapsed:   7.7s  VRAM: 2.25 GB
    48.1-  51.1s  elapsed:   8.1s  VRAM: 2.25 GB
    51.1-  54.1s  elapsed:   8.5s  VRAM: 2.25 GB
    54.1-  57.1s  elapsed:   8.9s  VRAM: 2.25 GB
    57.1-

In [1]:
from pathlib import Path

IGNORE = {'__pycache__', '.git', '.idea', '.vscode', 'node_modules', '.venv'}

def show_tree(path, prefix='', max_depth=4, depth=0):
    if depth > max_depth:
        return
    path = Path(path)
    if not path.exists():
        print(f'{path} — НЕ существует')
        return
    entries = sorted([e for e in path.iterdir() if e.name not in IGNORE])
    dirs = [e for e in entries if e.is_dir()]
    files = [e for e in entries if e.is_file()]
    for d in dirs:
        print(f'{prefix}📁 {d.name}/')
        show_tree(d, prefix + '  ', max_depth, depth + 1)
    for f in files:
        size = f.stat().st_size
        size_str = f'{size/1024/1024:.1f}M' if size > 1024*1024 else f'{size/1024:.0f}K' if size > 1024 else f'{size}B'
        print(f'{prefix}📄 {f.name}  ({size_str})')

show_tree('', max_depth=4)

📁 core/
  📄 __init__.py  (0B)
  📄 audio_io.py  (2K)
  📄 config.py  (2K)
  📄 converter.py  (544B)
  📄 residual.py  (3K)
  📄 solospeech.py  (7K)
  📄 vram.py  (950B)
📁 models/
  📁 SoloSpeech/
    📁 build/
      📁 bdist.win-amd64/
      📁 lib/
        📁 fastgeco/
        📁 geco/
        📁 solospeech/
    📁 checkpoints/
      📁 .cache/
        📁 huggingface/
      📄 .gitattributes  (2K)
      📄 compressor.ckpt  (585.6M)
      📄 config_compressor.json  (4K)
      📄 config_extractor.yaml  (659B)
      📄 config_tsr.yaml  (483B)
      📄 corrector.ckpt  (1252.8M)
      📄 extractor.pt  (5426.4M)
      📄 README.md  (595B)
      📄 tsr.pt  (1689.6M)
    📁 demo/
      📄 test2_solospeech.wav  (625K)
    📁 docs/
      📄 evaluation.md  (126B)
      📄 quick_use.md  (831B)
      📄 training.md  (1K)
    📁 scripts/
      📄 fast_test.py  (5K)
      📄 preprocess_long_audio.py  (30K)
      📄 test.py  (10K)
      📄 test_long_audio.py  (31K)
      📄 test_long_lowmem.py  (15K)
      📄 test_v2.py  (9K)
      📄 tes

## SoloSpeach

In [1]:
import torch
import torchvision
print(torch.__version__, torchvision.__version__)
print(torch.cuda.is_available())
import torchmetrics  # должно импортнуться без ошибок

2.7.1+cu118 0.22.1+cu118
True


c:\Users\matve\Documents\diplom\python_SSLR\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Итого

Все результаты — в `output/tse_out/`:
- `usef_tfgridnet_{8k,16k}.wav`
- `usef_sepformer_{8k,16k}.wav`
- `posneg_proposed.wav`
- `posneg_improved.wav`
- `meanflow.wav`